# 🔍 Hallucination Detection in Small Language Models

This notebook is your workspace for the **SMILES 2026** challenge.  Run the
cells from top to bottom to train and evaluate your hallucination detector.

### What you implement

| File | What to implement |
|------|-------------------|
| `aggregation.py` | Layer selection and token-pooling strategy |
| `probe.py` | Probe classifier (`nn.Module` subclass) |
| `splitting.py` | Train / validation / test split strategy |

### Fixed infrastructure (do not edit)

| File | Purpose |
|------|---------|
| `model.py` | LLM loader and hidden-state extractor |
| `evaluate.py` | Evaluation loop, summary table, JSON output |

## 0 · Environment Setup

Uncomment and run the block that matches your environment.

In [ ]:
# ── Google Colab ─────────────────────────────────────────────────────────
# Run this block once when opening the notebook in Colab for the first time.

# !git clone https://github.com/ahdr3w/SMILES-HALLUCINATION-DETECTION.git
# %cd SMILES-HALLUCINATION-DETECTION
# !pip install -r requirements.txt

# ── Local (terminal) ─────────────────────────────────────────────────────
# Run these commands in a terminal *before* launching Jupyter, then open
# this notebook from the cloned directory.
#
# git clone https://github.com/ahdr3w/SMILES-HALLUCINATION-DETECTION.git
# cd SMILES-HALLUCINATION-DETECTION
# python -m venv .venv
# source .venv/bin/activate      # Linux / macOS
# # .venv\Scripts\activate.bat   # Windows
# pip install -r requirements.txt
# jupyter notebook solution.ipynb

## 1 · Imports

In [ ]:
import time

import numpy as np
import pandas as pd
import torch

from aggregation import aggregate
from evaluate import print_summary, run_evaluation, save_predictions, save_results
from model import extract_hidden_states, get_model_and_tokenizer
from probe import HallucinationProbe
from splitting import split_data

## 2 · Configuration

In [ ]:
DATA_FILE   = "./data/dataset.csv"   # path to the dataset CSV
OUTPUT_FILE = "results.json"         # where to write the results summary
BATCH_SIZE  = 4                      # reduce to 1–2 if you run out of GPU memory

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device : {device}")
print(f"Data   : {DATA_FILE}")

## 3 · Load Dataset

In [ ]:
df = pd.read_csv(DATA_FILE)

# Build the text fed to the LLM: concatenation of prompt and response.
all_texts  = [f"{row['prompt']}{row['response']}" for _, row in df.iterrows()]
all_labels = np.array([int(float(h)) for h in df["label"]])

n_total = len(all_labels)
print(f"Loaded {n_total} samples  "
      f"({all_labels.sum()} hallucinated / {(all_labels == 0).sum()} truthful)")

## 4 · Feature Extraction

Hidden states are extracted **once** for the entire dataset.  The aggregation
function from `aggregation.py` is applied on-the-fly so only compact feature
vectors are kept in memory.  The resulting matrix `X` is reused across all
splits.

> ⚠️  This step loads the LLM and runs a forward pass — it may take a few
> minutes depending on your hardware.

In [ ]:
model, tokenizer = get_model_and_tokenizer()

t0 = time.time()
all_features, _ = extract_hidden_states(
    model,
    tokenizer,
    all_texts,
    device=device,
    batch_size=BATCH_SIZE,
    aggregate_fn=aggregate,   # applied in-place to save memory
)
extract_time = time.time() - t0
print(f"Extraction done in {extract_time:.1f} s")

# Stack feature vectors into a (N, feature_dim) NumPy matrix.
X = np.vstack([h.numpy() for h in all_features])   # shape: (N, feature_dim)
y = all_labels                                       # shape: (N,)

print(f"Feature matrix : {X.shape}  (feature_dim = {X.shape[1]})")

## 5 · Split Data

The splitting strategy is defined in **`splitting.py`** — edit that file to
change how the dataset is divided (random split, k-fold, group-aware, …).

`split_data` returns a list of `(idx_train, idx_val, idx_test)` tuples.  Pass
`df` as the second argument if your strategy needs per-sample metadata (e.g.
a `question` column for group-aware splits).

In [ ]:
splits = split_data(y, df)

print(f"Splits : {len(splits)} fold(s)")
for i, (tr, va, te) in enumerate(splits):
    print(f"  Fold {i + 1}: train={len(tr)}  "
          f"val={len(va) if va is not None else 'N/A'}  test={len(te)}")

## 6 · Evaluate

For every split the loop trains a majority-class baseline and your
`HallucinationProbe`, then reports Accuracy, F1, and AUROC.

In [ ]:
fold_results = run_evaluation(splits, X, y, HallucinationProbe)

## 7 · Summary & Save Results

In [ ]:
print_summary(fold_results, X.shape[1], len(X), extract_time)
save_results(fold_results, X.shape[1], len(X), extract_time, OUTPUT_FILE)

## 8 · Predict on Test Set

Run the probe on the held-out competition test file (`data/test.csv`) whose
`label` column contains only null values.  The probe is fitted on the
**training and validation data only** — the evaluation test set remains held out
so that the metrics from Section 6 remain an honest estimate of performance.

Predictions are saved to `PREDICTIONS_FILE` (default `predictions.csv`).

In [ ]:
TEST_FILE        = "./data/test.csv"   # competition test set (labels are null)
PREDICTIONS_FILE = "predictions.csv"   # output file with predicted labels

# ── Load test data ────────────────────────────────────────────────────────
df_test      = pd.read_csv(TEST_FILE)
test_texts   = [f"{row['prompt']}{row['response']}" for _, row in df_test.iterrows()]
test_ids     = df_test["id"].tolist()

print(f"Test set loaded: {len(test_texts)} samples")

# ── Extract features ────────────────────────────────────────────────────
test_features, _ = extract_hidden_states(
    model,
    tokenizer,
    test_texts,
    device=device,
    batch_size=BATCH_SIZE,
    aggregate_fn=aggregate,
)
X_test = np.vstack([h.numpy() for h in test_features])  # (n_test, feature_dim)

# ── Fit final probe on training + validation data only ──────────────────
# Collect the union of all train and validation indices across every split.
# For a single split this excludes idx_test; for k-fold every sample appears
# in a training fold, so all samples are used (same as fitting on X, y).
idx_non_test = np.unique(np.concatenate([
    np.concatenate([idx_tr, idx_va]) if idx_va is not None else idx_tr
    for idx_tr, idx_va, _ in splits
]))
final_probe = HallucinationProbe()
final_probe.fit(X[idx_non_test], y[idx_non_test])

# ── Predict and save ────────────────────────────────────────────────────
save_predictions(final_probe, X_test, test_ids, PREDICTIONS_FILE)
